<a href="https://colab.research.google.com/github/Addyk-24/Movie-Classification-Model/blob/main/Movie_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
# genre_list = []
# movie_name_list = []
# movie_desc_list = []

# with open("train_data.txt",'r') as file:
#   for line in file:
#     parts = line.strip().split(':::')
#     if(len(parts) == 4):
#       index,movie_name,genre,movie_desc = parts
#       movie_desc_list.append(movie_desc)
#       movie_name_list.append(movie_name)
#       genre_list.append(genre)

#     # print("Movie name: ",movie_name)
#     # print("Genre: ",genre)
# ds = {
#         "Movie Name": movie_name_list,
#         "Movie Description": movie_desc_list,
#         "Movie Genre": genre_list
#       }
# movie_ds = pd.DataFrame(ds)


In [ ]:
# Tokenizing the texts
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

movie_ds = pd.read_csv('/kaggle/input/movie-ds/movie_ds.csv')


texts = movie_ds['Movie Description'].astype(str).values

# Tokenize the text - CONSISTENT PARAMETERS
vocab_size = 10000  # Increased vocabulary size
max_length = 150    # Keep longer sequences for movie descriptions
embedding_dim = 100  # Increased embedding dimension

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

y = movie_ds.iloc[:, -1]

# One-hot encode the labels
y_ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [0])], remainder='passthrough')
y = y_ct.fit_transform(y.values.reshape(-1, 1)).toarray()

In [ ]:
# Improved model architecture
ann = tf.keras.models.Sequential()

# Embedding layer with matching parameters
ann.add(tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length))

# First LSTM layer
ann.add(tf.keras.layers.LSTM(units=128, dropout=0.3, recurrent_dropout=0.2))

# Dense hidden layer
ann.add(tf.keras.layers.Dense(units=64, activation='relu'))
ann.add(Dropout(0.3))


# Output layer (27 genres)
ann.add(tf.keras.layers.Dense(units=27, activation='softmax'))

# Compile
ann.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    mode='min',
    verbose=1
)


# Model summary
ann.summary()

# Training
history = ann.fit(
    X, y,
    batch_size=64,  # Smaller batch size for better generalization
    epochs=50,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

### Saved Model After Training

In [ ]:
# from tensorflow.keras.models import load_model
# loaded_model = load_model('movie_genre_classifier.keras')

In [ ]:
X_test = pd.read_csv('new_test_ds.csv')

X_test = X_test['Movie Description'].astype(str).values

# Step 2: Tokenize and pad the test data
test_sequences = tokenizer.texts_to_sequences(X_test)
X_test_padded = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

# Step 3: Make predictions on the TEST data
predictions_probs = ann.predict(X_test_padded)  # This gives probabilities
predictions = np.argmax(predictions_probs, axis=1)  # Convert to class indices

# Step 4: Load the test solutions
test_ds = pd.read_csv('test_ds_solution.csv')
y_test = test_ds['Movie Genre']

y_test = y_ct.transform(y_test.values.reshape(-1, 1)).toarray()


In [ ]:
# Prediction
predictions_probs = ann.predict(X_test_padded, batch_size=64, verbose=1)
predictions = np.argmax(predictions_probs, axis=1)
print(f"\nFull predictions shape: {predictions.shape}")

In [ ]:
# Convert y_test from one-hot encoding to class indices
y_test_indices = np.argmax(y_test, axis=1)

In [ ]:
# Calculate metrics
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

cm = confusion_matrix(y_test_indices, predictions)
accuracy = accuracy_score(y_test_indices, predictions)

print(f"\n🎯 Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\nConfusion Matrix shape: {cm.shape}")

In [ ]:
print("\n📊 Classification Report:")
print(classification_report(y_test_indices, predictions, target_names=genre_names))

# Show per-genre accuracy
print("\n📈 Per-Genre Accuracy:")
for i, genre in enumerate(genre_names):
    mask = y_test_indices == i
    if mask.sum() > 0:
        genre_acc = (predictions[mask] == i).sum() / mask.sum()
        print(f"{genre:20s}: {genre_acc:.2%} ({mask.sum()} samples)")

In [ ]:

# Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(history.history['loss'], label='Training Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_title('Model Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(history.history['accuracy'], label='Training Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax2.set_title('Model Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
print(f"\nFinal Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
